In [3]:
#importing libraries
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [ ]:
# loading dataset wand initial EDA with SweetViz
insurance = pd.read_csv(""Week 2/4-7 In Class Assignment.ipynb")
insurance.head()

#visualizing the data with SweetViz
report = sv.analyze(insurance)
report.show_html('insrurance_report.html')


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report insrurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Age and BMI are fairly normally distributed, while most people have 0–2 children. The dataset is balanced by sex and region, but most individuals are non-smokers. BMI and smoking status likely have strong impact on charges.

In [6]:
#encoding categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [11]:
# preparing the data for modeling
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

# splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scaling the features (don't need for RF but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#baseline linear regression model
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

#baseline random forest model
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf) 

# print baseline results
print(f'Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Baseline Random Forest MSE: {mse_rf:.2f}, R2: {r2_rf:.2f}')



Baseline Linear Regression MSE: 33566439.74, R2: 0.78
Baseline Random Forest MSE: 22179121.67, R2: 0.86


Random Forest performs better than Linear Regression (higher R^2, lower MSE), so it captures patterns in the data better.

In [8]:
# CV with baseline models with r2 as the scoring metric
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='r2')
print(f'Baseline Linear Regression CV R2 Scores: {cv_scores_lr}')
print(f'Baseline Random Forest CV R2 Scores: {cv_scores_rf}')   





Baseline Linear Regression CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]
Baseline Random Forest CV R2 Scores: [0.81530553 0.89851976 0.79006114 0.78263259 0.82825938]


In [9]:
# gridsearchCV for hyperparameter tuning of random forest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['log2', 'sqrt']
}           

grid_search_rf = GridSearchCV(estimator=baseline_rf, 
                              param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)    
print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')


Best RF Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Best RF CV R2 Score: 0.84


Best parameters from GridSearchCV:
n_estimators: 200
max_depth: 10
min_samples_split: 5
max_features: log2

In [10]:
# examine top 20 ranked parameters from grid search
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
11           200        10                  5         log2         0.841202
3            200      None                  5         log2         0.840662
19           200        20                  5         log2         0.840662
10           100        10                  5         log2         0.840452
8            100        10                  2         log2         0.840383
9            200        10                  2         log2         0.839967
18           100        20                  5         log2         0.839195
2            100      None                  5         log2         0.839195
17           200        20                  2         log2         0.836639
1            200      None                  2         log2         0.836593
16           100        20                  2         log2         0.835721
0            100      None                  2         log2

The top 20 models had very similar scores, but I would choose the model with 200 estimators, max depth 10, min samples split 5, and max features log2 because it had the best overall CV R^2 score. It also uses a limited depth, which helps control overfitting while still giving strong performance.